# PrismaCoder-3B - End-to-End QLoRA Fine-tuning on Kaggle

Trains **PrismaCoder-3B** (Qwen2.5-Coder-3B-Instruct base) on **PrismaForge** - the open, execution-verified Text-to-Prisma corpus - using QLoRA. Every step runs top-to-bottom without manual edits.

**Pipeline**: install -> configure -> fetch repo + data -> load 4-bit model -> tokenize -> QLoRA train -> static gate eval -> save checkpoints -> merge adapter -> push to HF -> export GGUF.

> Hardware note: default `MAX_TRAIN_SAMPLES=4000` completes in a 12h T4 session. Set `MAX_TRAIN_SAMPLES=0` for the full 120k-record run (8x A100 recipe: `training/configs/full_ft.yaml`).


In [ ]:
# Cell 1 - install pinned dependencies (Kaggle ships torch; add the trainer stack)
%pip install -q -U transformers==4.46.3 peft==0.13.2 trl==0.12.2 \
    datasets==3.1.0 accelerate==1.0.1 bitsandbytes==0.44.1 \
    huggingface-hub==0.26.5 sentencepiece tensorboard \
    'PyYAML>=6.0' requests

import importlib, sys
for pkg in ['transformers', 'peft', 'trl', 'datasets', 'accelerate', 'bitsandbytes']:
    print(pkg, importlib.import_module(pkg).__version__)


In [ ]:
# Cell 2 - configuration + secrets
import os

SEED = 20260830
BASE_MODEL = 'Qwen/Qwen2.5-Coder-3B-Instruct'
HF_USERNAME = 'prismacoder'          # change to your org when pushing
DATASET_REPO = f'{HF_USERNAME}/PrismaForge'
OUTPUT_REPO = f'{HF_USERNAME}/PrismaCoder-3B-QLoRA'
MAX_TRAIN_SAMPLES = 4000             # 0 = full 120k (multi-GPU recommended)
MAX_VAL_SAMPLES = 200
MAX_SEQ_LEN = 4096
EPOCHS = 3
LR = 1e-4                            # adapter LR (Table 12.1 S1)
WORK_DIR = '/kaggle/working/prismacoder'

# HF token: Kaggle secret 'HF_TOKEN' -> falls back to anonymous (public data only)
def _get_secret(name: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name, '')

HF_TOKEN = _get_secret('HF_TOKEN')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF token loaded from Kaggle secrets')
else:
    print('No HF token found: public repos only; push/gated-download steps will skip')


In [ ]:
# Cell 3 - fetch the PrismaCoder repository (package code for data + validation)
import os, shutil, subprocess

REPO_URL = 'https://github.com/prismacoder/PrismaCoder.git'
if not os.path.exists(WORK_DIR):
    if shutil.which('git') is None:
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'git'], check=False)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, WORK_DIR], check=True)
os.chdir(WORK_DIR)
sys_path_added = str(WORK_DIR)
import sys
if sys_path_added not in sys.path:
    sys.path.insert(0, sys_path_added)
from prismacoder import __version__ as pc_version
print('PrismaCoder package:', pc_version)


In [ ]:
# Cell 4 - download PrismaForge (HF hub) or build a mini-corpus deterministically
import os
from pathlib import Path

DATA_DIR = Path('/kaggle/working/data'); DATA_DIR.mkdir(exist_ok=True)
train_file = DATA_DIR / 'train.jsonl'
val_file = DATA_DIR / 'validation.jsonl'

def download_from_hub() -> bool:
    try:
        from huggingface_hub import hf_hub_download
        for fname, target in (('train.jsonl', train_file), ('validation.jsonl', val_file)):
            p = hf_hub_download(DATASET_REPO, fname, repo_type='dataset',
                                local_dir=str(DATA_DIR))
            assert Path(p).exists()
        return True
    except Exception as e:
        print('hub download unavailable:', type(e).__name__, '-', e)
        return False

if not download_from_hub():
    print('building a deterministic mini-corpus with the synthetic generator...')
    from prismacoder.synthetic.generator import SyntheticGenerator
    gen = SyntheticGenerator(SEED)
    n = MAX_TRAIN_SAMPLES or 4000
    recs = gen.generate(max(n, 600))
    gen.attach_reasoning(recs, fraction=0.10)
    from prismacoder.io_utils import write_jsonl
    write_jsonl(train_file, recs[:-100])
    write_jsonl(val_file, recs[-100:])

from prismacoder.io_utils import count_lines
print('train records:', count_lines(train_file), '| val records:', count_lines(val_file))


In [ ]:
# Cell 5 - load tokenizer + 4-bit (nf4) model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    torch_dtype=torch.bfloat16,
    attn_implementation='sdpa',       # flash-attn wheels are not on Kaggle by default
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
print(model)


In [ ]:
# Cell 6 - tokenize: chat format with completion-only loss (training/trainer/data.py)
from datasets import Dataset
from training.trainer.data import load_sft_records, to_chat_example

train_records, val_records = load_sft_records(str(train_file), str(val_file),
                                              limit=MAX_TRAIN_SAMPLES or None)
val_records = val_records[:MAX_VAL_SAMPLES]

def tokenize_record(rec):
    ex = to_chat_example(rec, tokenizer)
    return {'input_ids': ex['input_ids'], 'labels': ex['labels']}

train_ds = Dataset.from_list([tokenize_record(r) for r in train_records])
val_ds = Dataset.from_list([tokenize_record(r) for r in val_records]) if val_records else None
print(train_ds)


In [ ]:
# Cell 7 - QLoRA adapter (r=16, alpha=32, all linear projections)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
lora = LoraConfig(
    task_type='CAUSAL_LM',
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()


In [ ]:
# Cell 8 - train with TRL SFTTrainer (packing, NEFTune 5, cosine 3% warmup)
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir='/kaggle/working/checkpoints',
    run_name='prismacoder-3b-qlora-kaggle',
    seed=SEED,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,      # effective batch 16 on 1 GPU
    bf16=False,                          # T4 has no bf16; set True on A100/L4
    fp16=torch.cuda.is_available() and 'A100' not in torch.cuda.get_device_name(0),
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    max_grad_norm=1.0,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy='steps' if val_ds is not None else 'no',
    eval_steps=100,
    save_strategy='steps', save_steps=200, save_total_limit=3,
    report_to=['tensorboard'],
    max_seq_length=MAX_SEQ_LEN,
    packing=True,
    neftune_noise_alpha=5,
    dataset_text_field=None,             # pre-tokenized dataset
    remove_unused_columns=False,
)

class LossMaskCollator:
    def __init__(self, pad_id): self.pad_id = pad_id
    def __call__(self, feats):
        import torch
        from torch.nn.utils.rnn import pad_sequence
        ids = [torch.tensor(f['input_ids']) for f in feats]
        lbs = [torch.tensor(f['labels']) for f in feats]
        ids = pad_sequence(ids, batch_first=True, padding_value=self.pad_id)
        lbs = pad_sequence(lbs, batch_first=True, padding_value=-100)
        return {'input_ids': ids, 'labels': lbs,
                'attention_mask': (ids != self.pad_id).long()}

trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
                     eval_dataset=val_ds, data_collator=LossMaskCollator(tokenizer.pad_token_id))
trainer.train()


In [ ]:
# Cell 9 - static gate evaluation on held-out records (no Node needed on Kaggle)
from prismacoder.verification.verifier import LightValidator
import re

light = LightValidator()
model.eval()

def generate_artifact(rec, max_new_tokens=768):
    from prismacoder.constants import SYSTEM_PROMPT
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': rec['task']['instruction']}
                ]
    schema = rec['task']['context']['schema']
    messages[1]['content'] += '\n\n```prisma\n' + schema + '\n```'
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

passed = total = 0
for rec in val_records[:25]:
    text = generate_artifact(rec)
    verdict = light.validate(rec['task']['context']['schema'], text)
    total += 1
    passed += verdict.status != 'failed'
print(f'static validation pass: {passed}/{total}')


In [ ]:
# Cell 10 - save checkpoints + tokenizer to /kaggle/working
ckpt_dir = '/kaggle/working/prismacoder-3b-qlora'
trainer.save_model(ckpt_dir)
tokenizer.save_pretrained(ckpt_dir)
import os
print(sorted(os.listdir(ckpt_dir)))


In [ ]:
# Cell 11 - merge adapter into the base model
from peft import PeftModel

merged_dir = '/kaggle/working/prismacoder-3b-merged'
del model; del trainer; torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16,
                                            device_map='cpu', trust_remote_code=True)
peft_model = PeftModel.from_pretrained(base, ckpt_dir)
merged = peft_model.merge_and_unload()
merged.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)
print('merged ->', merged_dir)


In [ ]:
# Cell 12 - push model + dataset to the HuggingFace Hub
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(OUTPUT_REPO, repo_type='model', private=True, exist_ok=True)
    api.upload_large_folder(repo_id=OUTPUT_REPO, repo_type='model', folder_path=merged_dir)
    print('pushed:', OUTPUT_REPO)
else:
    print('skipped (no HF token) - merged model kept at', merged_dir)


In [ ]:
# Cell 13 - export GGUF (llama.cpp convert + q8_0 quantize)
import subprocess, sys, os

LC_DIR = '/kaggle/working/llama.cpp'
if not os.path.exists(LC_DIR):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/ggml-org/llama.cpp', LC_DIR], check=True)
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                    f'{LC_DIR}/requirements.txt'], check=False)
    gguf_out = '/kaggle/working/PrismaCoder-3B-Q8_0.gguf'
    subprocess.run([sys.executable, f'{LC_DIR}/convert_hf_to_gguf.py', merged_dir,
                    '--outfile', gguf_out, '--outtype', 'f16'], check=True)
    print('GGUF ->', gguf_out)
except Exception as e:
    print('GGUF export failed (non-fatal):', e)


## Next steps

- Evaluate the merged model on **PrismaBench** with Node gates: `notebooks/03_evaluation.ipynb` (or `benchmark/evaluator/evaluate.py --full-gates`).
- Continue with **DPO** on verifier-labeled pairs: `training/configs/dpo_qlora.yaml`.
- The full 120k-record, 8x A100 recipe lives in `training/configs/full_ft.yaml`.
